# 07_01 — Counterfactual Backtest: All Strategies Over the Validation Period

## 1. Objective and Temporal Framework

This notebook runs a **counterfactual backtest** — the core evaluation of the DSS.
We simulate what total energy procurement cost would have been under each strategy,
applied to the same historical validation period with the same market prices.

### Strategies compared
| Strategy | Description |
|---|---|
| `spot_only` | Buy 100% at daily spot price — no hedging, no flexibility |
| `static_hedge` | Pre-buy 70% at front-month futures; remainder at spot |
| `heuristic_policy` | DSS rule engine — hedge or shift only when tail-risk signals fire |
| `rl_policy` | DSS RL engine — actions learned from reward signal |

### Temporal causality guarantee
The OMIP Day-Ahead auction publishes the t+1 spot price at **13:00 on day t**.
This means:
- **t+1 price is KNOWN** at decision time — treated as realized market input.
- **t+2, t+3 and beyond are UNKNOWN** — the DSS uses quantile predictions (q50, q90).
- All lag and rolling features use `pandas.shift()` — strictly past information.
- Train / validation splits are **chronological** with no row overlap.

No look-ahead bias is introduced at any stage of the pipeline.

**Source data:** `data/processed/train_features.csv` · `data/processed/validation_features.csv`

**Core modules:** `src/backtesting/simulate_baseline.py` · `simulate_policy.py` · `simulate_rl_policy.py`

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = Path('../../').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

train_df      = pd.read_csv(project_root / 'data/processed/train_features.csv')
validation_df = pd.read_csv(project_root / 'data/processed/validation_features.csv')

print(f'Train:      {train_df.shape[0]:,} rows  |  {train_df["date"].min()} -> {train_df["date"].max()}')
print(f'Validation: {validation_df.shape[0]:,} rows  |  {validation_df["date"].min()} -> {validation_df["date"].max()}')

## 2. Build Policy Inputs

The `prepare_policy_inputs()` bridge merges quantile forecasts (trained on `train_df`,
evaluated on `validation_df`) with the market variables needed by both decision engines.
Columns `q_0.5` and `q_0.9` are **forecast inputs** to the policy — not realized costs.

In [ ]:
from src.models.train_model import train_quantile_suite
from src.decision.policy_inputs import prepare_policy_inputs, summarize_policy_inputs

quantile_output  = train_quantile_suite(train_df=train_df, eval_df=validation_df)
policy_inputs_df = prepare_policy_inputs(validation_df, quantile_output.results)

print('Policy inputs overview:')
display(summarize_policy_inputs(policy_inputs_df))
display(policy_inputs_df[['date', 'Spot_Price_SPEL', 'Future_M1_Price', 'q_0.5', 'q_0.9']].head(8))

## 3. Simulate All Four Strategies

Each simulator converts daily procurement decisions into **counterfactual cost** rows.
The realized `Spot_Price_SPEL` is used as the cost basis (t+1 price, known at decision time).

### Factory energy model (from `src/config/constants.py`)
```
energy_consumed = base_load (0.30) + variable_load (0.70) x production_level
total_cost      = energy_consumed x unit_price
```
- `do_nothing`       -> unit_price = spot
- `buy_m1_future`    -> unit_price = futures (locked in advance)
- `shift_production` -> reduced load + shift penalty per MWh

In [ ]:
from src.backtesting.simulate_baseline import (
    simulate_spot_only_baseline,
    simulate_static_hedge_baseline,
)
from src.backtesting.simulate_policy import simulate_policy_strategy
from src.backtesting.simulate_rl_policy import simulate_rl_policy_strategy
from src.decision.heuristic_policy import apply_heuristic_policy
from src.rl.train_rl_agent import train_q_learning_agent
from src.decision.rl_policy import apply_rl_policy

spot_only_df    = simulate_spot_only_baseline(policy_inputs_df)
static_hedge_df = simulate_static_hedge_baseline(policy_inputs_df)

heuristic_policy_df = apply_heuristic_policy(policy_inputs_df)
heuristic_sim_df    = simulate_policy_strategy(heuristic_policy_df)

rl_artifacts        = train_q_learning_agent(policy_inputs_df)
rl_policy_artifacts = apply_rl_policy(agent=rl_artifacts.agent, policy_inputs_df=policy_inputs_df)

# decisions_df only contains date + action columns; merge price data before simulation
rl_decisions_with_prices = rl_policy_artifacts.decisions_df.merge(
    policy_inputs_df[["date", "Spot_Price_SPEL", "Future_M1_Price"]],
    on="date", how="left",
)
rl_sim_df = simulate_rl_policy_strategy(rl_decisions_with_prices)

print('All four strategies simulated.')
for name, df in [('spot_only', spot_only_df), ('static_hedge', static_hedge_df),
                  ('heuristic', heuristic_sim_df), ('rl_policy', rl_sim_df)]:
    print(f'  {name:15s}: {len(df)} rows  |  total_cost = {df["total_cost"].sum():.2f}')

## 4. Daily Cost Table

The combined view shows all four cost columns aligned by date.
Differences on individual days reveal the economic impact of each decision.

In [ ]:
from src.backtesting.compare_strategies import compare_daily_costs

daily_costs_df = compare_daily_costs([spot_only_df, static_hedge_df, heuristic_sim_df, rl_sim_df])

display(daily_costs_df.head(12).round(2))
print(f'Date range: {daily_costs_df["date"].min()} -> {daily_costs_df["date"].max()}')

## 5. Aggregate Summary

Key metrics across the full validation period:
- **total_cost** — sum of daily procurement costs (lower is better)
- **average_unit_cost** — EUR per MWh procured (normalised by volume)
- **daily_cost_volatility** — std dev of daily cost (stability proxy; lower is better)
- **max_daily_cost** — worst-case single day (tail-risk exposure)
- **savings_vs_spot_only** — absolute savings compared to the do-nothing baseline
- **savings_share_vs_spot_only** — savings as a fraction of spot-only cost

In [ ]:
from src.backtesting.compare_strategies import build_strategy_comparison_report

report = build_strategy_comparison_report(
    [spot_only_df, static_hedge_df, heuristic_sim_df, rl_sim_df],
    reference_strategy_name='spot_only',
)

print('Full summary (sorted by total cost):')
display(report['summary_table'].round(4))

print('\nSavings vs spot-only:')
cols = ['strategy_name', 'total_cost', 'savings_vs_spot_only',
        'savings_share_vs_spot_only', 'daily_cost_volatility']
available = [c for c in cols if c in report['summary_vs_reference'].columns]
display(report['summary_vs_reference'][available].round(4))

## 6. Heuristic Policy Action Distribution

High `do_nothing` frequency is expected — the heuristic engine hedges only when the
tail-risk premium over the futures price exceeds the configured threshold.
This selectivity is a feature, not a bug: it avoids unnecessary transaction costs.

In [ ]:
from src.decision.heuristic_policy import summarize_policy_actions

print('Heuristic policy action mix:')
display(summarize_policy_actions(heuristic_policy_df))

print('\nSample of heuristic decisions:')
sample_cols = ['date', 'recommended_action', 'decision_reason', 'tail_vs_future_abs']
avail = [c for c in sample_cols if c in heuristic_policy_df.columns]
display(heuristic_policy_df[avail].head(10))